In [4]:
!pip -q install requests beautifulsoup4 pandas openpyxl


[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import re
import time
import sqlite3
from urllib.parse import urljoin, urlparse, parse_qs

In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
BASE = "https://yoruba.unl.edu/"
SLEEP_SECONDS = 1.0

HEADERS = {
    "User-Agent": "HeriotWatt-Dubai-ResearchBot/1.0 (academic project; contact: temisolaolajide@gmail.com)"
}

session = requests.Session()
session.headers.update(HEADERS)

In [8]:
def get_soup(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

In [11]:
PARTS = {
    1: {
        "code": "1a",
        "theme": "humility, self-control, self-knowledge, self-respect, self-restraint"
    },
    2: {
        "code": "1b",
        "theme": "perspicaciousness, reasonableness, sagacity, savoir-faire, wisdom, worldly wisdom"
    },
    3: {
        "code": "1c",
        "theme": "cageyness, caution, moderation, patience, prudence"
    },
    4: {
        "code": "1d",
        "theme": "perseverance, industry, resilience, self-confidence, self-reliance, resourcefulness, daring"
    },
    5: {
        "code": "1e",
        "theme": "consistency; honesty, openness, plain speaking, reliability"
    },
    6: {
        "code": "1f",
        "theme": "consideration, kindness, thoughtfulness"
    }
}

def index_url(part_code: str) -> str:
    # l=0 is the alphabetical index page
    return f"{BASE}yoruba.php-text={part_code}&view=0&uni=0&l=0.htm"


In [12]:
def extract_letter_pages_from_index(part_id: int, part_code: str):
    """
    Returns a list of dicts:
    [
      {"letter": "A", "entries_text": "220 entries", "url": "...&l=1.htm"},
      {"letter": "B", "entries_text": "55 entries", "url": "...&l=2.htm"},
      ...
    ]
    """
    url = index_url(part_code)
    soup = get_soup(url)

    letter_pages = []
    for a in soup.select("a[href]"):
        text = a.get_text(" ", strip=True)
        href = a.get("href", "").strip()
        if not href:
            continue

        # On the site, the clickable text next to letters is often like "220 entries"
        # Some pages use exactly "entries" wording.
        if "entries" not in text.lower():
            continue

        full_url = urljoin(url, href)

        # letter is usually the previous sibling text or near the link.
        # We'll try to infer it from the surrounding table row/cell.
        letter = ""
        parent_text = a.find_parent().get_text(" ", strip=True) if a.find_parent() else ""
        # Example parent_text might be: "A 220 entries"
        m = re.match(r"^([A-Za-zỌẸṢọbaẹṣgbGB]+)\s+.*entries", parent_text)
        if m:
            letter = m.group(1).strip()

        letter_pages.append({
            "part_id": part_id,
            "part_code": part_code,
            "letter": letter,
            "entries_text": text,
            "url": full_url
        })

    # Deduplicate (sometimes repeated)
    seen = set()
    out = []
    for item in letter_pages:
        if item["url"] not in seen:
            seen.add(item["url"])
            out.append(item)

    return out


In [13]:
def looks_yoruba(line: str) -> bool:
    # Strong signal: Yoruba diacritics
    return bool(re.search(r"[ẹ̀ẹọ́ọ̀ṣṢńáéíóúàèìòùẸỌṢ]", line)) and not line.startswith("(")

def parse_letter_page(letter_page_meta: dict):
    url = letter_page_meta["url"]
    soup = get_soup(url)

    text = soup.get_text("\n", strip=True)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # Fallback: try to find the letter in the page header if missing
    letter = letter_page_meta.get("letter") or ""
    if not letter:
        # Often the page shows "Part X: ..." then the letter on the next line
        for i, ln in enumerate(lines[:80]):
            if ln.startswith("Part ") and i + 1 < len(lines):
                possible_letter = lines[i + 1].strip()
                # Letters can be A, B, Ẹ, Gb, Ọ, Ṣ, etc.
                if len(possible_letter) <= 3:
                    letter = possible_letter
                break

    records = []
    i = 0
    while i < len(lines):
        ln = lines[i]

        # Skip obvious navigation noise
        if ln in {"Contents", "Display Settings", "Font Settings", "About", "Main Contents"}:
            i += 1
            continue

        if looks_yoruba(ln):
            yoruba = ln
            english_literal = ""
            interpretation = ""
            notes = []

            # Next line is often literal English translation
            if i + 1 < len(lines) and not looks_yoruba(lines[i + 1]) and not lines[i + 1].startswith("Part "):
                english_literal = lines[i + 1]
                i += 1

            # Consume until next Yoruba proverb starts
            j = i + 1
            while j < len(lines) and not looks_yoruba(lines[j]) and not lines[j].startswith("Part "):
                s = lines[j]
                if s.startswith("(") and s.endswith(")"):
                    # If multiple parenthetical lines exist, append
                    if interpretation:
                        interpretation += " " + s.strip("()")
                    else:
                        interpretation = s.strip("()")
                else:
                    notes.append(s)
                j += 1

            records.append({
                "part_id": letter_page_meta["part_id"],
                "part_code": letter_page_meta["part_code"],
                "part_theme": PARTS[letter_page_meta["part_id"]]["theme"],
                "letter": letter,
                "yoruba": yoruba,
                "english_literal": english_literal,
                "interpretation_gold": interpretation,
                "notes": " ".join(notes).strip(),
                "source_url": url
            })

            i = j
        else:
            i += 1

    return records


In [14]:
all_records = []

for part_id, meta in PARTS.items():
    part_code = meta["code"]

    time.sleep(SLEEP_SECONDS)
    letter_pages = extract_letter_pages_from_index(part_id, part_code)
    print(f"Part {part_id} ({part_code}) -> {len(letter_pages)} letter pages found")

    for lp in letter_pages:
        time.sleep(SLEEP_SECONDS)
        all_records.extend(parse_letter_page(lp))

df = pd.DataFrame(all_records)
print("Total proverbs scraped:", len(df))
df.head(10)


Part 1 (1a) -> 22 letter pages found
Part 2 (1b) -> 20 letter pages found
Part 3 (1c) -> 23 letter pages found
Part 4 (1d) -> 17 letter pages found
Part 5 (1e) -> 20 letter pages found
Part 6 (1f) -> 13 letter pages found
Total proverbs scraped: 3286


,part_id,part_code,part_theme,letter,yoruba,english_literal,interpretation_gold,notes,source_url
0,1,1a,"humility, self-control, self-knowledge, self-r...",A,A di gàárì sílẹ̀ ewúrẹ́ ńyọjú; ẹrù ìran rẹ̀ ni?,"We prepare the saddle, and the goat presents i...",Goats that know their place do not offer their...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
1,1,1a,"humility, self-control, self-knowledge, self-r...",A,This is a variant of A gbé gàárì ọmọ ewúrẹ́ ńr...,,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
2,1,1a,"humility, self-control, self-knowledge, self-r...",A,A fi ọ́ jọba ò ńṣàwúre o fẹ́ jẹ Ọlọ́run ni?,"You have been crowned a king, and yet you make...",Being crowned a king is about the best fortune...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
3,1,1a,"humility, self-control, self-knowledge, self-r...",A,"A fijó gba Awà; a fìjà gba Awà; bí a ò bá jó, ...",,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
4,1,1a,"humility, self-control, self-knowledge, self-r...",A,By dancing we take possession of Awà; through ...,(Why make a huge production of a matter that i...,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
5,1,1a,"humility, self-control, self-knowledge, self-r...",A,A gbé gàárì ọmọ ewurẹ ńrojú; kì í ṣe ẹrù àgùntàn.,We lift a saddle and the goat (kin) scowls; it...,"The goat has no cause to scowl, because no one...",,https://yoruba.unl.edu/yoruba.php-text=1a&view...
6,1,1a,"humility, self-control, self-knowledge, self-r...",A,This is a variant of A di gàárì sílẹ̀ . . .,,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
7,1,1a,"humility, self-control, self-knowledge, self-r...",A,A kì í bá ọba pàlà kí ọkọ́ ọba má ṣàn-ánni lé...,One does not share a farm boundary with a king...,One should be cautious in dealing with people ...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
8,1,1a,"humility, self-control, self-knowledge, self-r...",A,A kì í bínú ààtàn ká dalẹ̀ sígbẹ̀ẹ́.,One does not get angry with the rubbish dump a...,One should not act in unreasonable and harmful...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
9,1,1a,"humility, self-control, self-knowledge, self-r...",A,A kì í bínú orí ká fi fìlà dé ìbàdí.,One does not get angry with one's head and the...,Do not cut your nose to spite your face.,,https://yoruba.unl.edu/yoruba.php-text=1a&view...


In [15]:
output_path = "dissertation_yoruba_proverbs.xlsx"

# tidy column order
cols = [
    "part_id", "part_code", "part_theme", "letter",
    "yoruba", "english_literal", "interpretation_gold", "notes",
    "source_url"
]
df = df[cols].copy()

df.to_excel(output_path, index=False)
output_path


'dissertation_yoruba_proverbs.xlsx'

In [16]:
output_path_parts = "yoruba_proverbs_by_part.xlsx"

with pd.ExcelWriter(output_path_parts, engine="openpyxl") as writer:
    # Optional: include the full dataset as the first sheet
    df.to_excel(writer, sheet_name="ALL", index=False)

    # Create one sheet per part_id
    for part_id in sorted(df["part_id"].unique()):
        df_part = df[df["part_id"] == part_id].copy()

        # Sheet names must be <= 31 chars and cannot contain


In [17]:
import hashlib

def make_entry_id(row):
    key = f"{row['part_id']}|{row['letter']}|{row['yoruba']}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]

df = df.copy()
df["entry_id"] = df.apply(make_entry_id, axis=1)

# Move entry_id to the front
cols = ["entry_id"] + [c for c in df.columns if c != "entry_id"]
df = df[cols]


In [18]:
df.to_csv("yoruba_proverbs.csv", index=False, encoding="utf-8")

In [19]:
df.to_parquet("yoruba_proverbs.parquet", index=False)

In [20]:
pd.read_parquet("yoruba_proverbs.parquet").head()

,entry_id,part_id,part_code,part_theme,letter,yoruba,english_literal,interpretation_gold,notes,source_url
0,a1845a38747df71e,1,1a,"humility, self-control, self-knowledge, self-r...",A,A di gàárì sílẹ̀ ewúrẹ́ ńyọjú; ẹrù ìran rẹ̀ ni?,"We prepare the saddle, and the goat presents i...",Goats that know their place do not offer their...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
1,d1028fe117c17f87,1,1a,"humility, self-control, self-knowledge, self-r...",A,This is a variant of A gbé gàárì ọmọ ewúrẹ́ ńr...,,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
2,a6223c8a370d0dd5,1,1a,"humility, self-control, self-knowledge, self-r...",A,A fi ọ́ jọba ò ńṣàwúre o fẹ́ jẹ Ọlọ́run ni?,"You have been crowned a king, and yet you make...",Being crowned a king is about the best fortune...,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
3,c088fcb6062ca2a7,1,1a,"humility, self-control, self-knowledge, self-r...",A,"A fijó gba Awà; a fìjà gba Awà; bí a ò bá jó, ...",,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...
4,c35f4b8d772e1ba1,1,1a,"humility, self-control, self-knowledge, self-r...",A,By dancing we take possession of Awà; through ...,(Why make a huge production of a matter that i...,,,https://yoruba.unl.edu/yoruba.php-text=1a&view...


In [21]:
pd.read_csv("yoruba_proverbs.csv").shape

(3286, 10)

In [22]:
df.to_csv("yoruba_proverbs.csv", index=False, encoding="utf-8-sig")
